# Business Entity Resolution

Amazon ML Challenge 2026 solution. The task is to match each Source 1 (reference) business record to the records
for the same real-world business in Source 2 and Source 3, across the US, India and France. The score is a macro
F0.5 over Source 1 entities, so precision is weighted twice as much as recall.

## Pipeline overview

1. **Data preparation.** Clean and normalize business names, addresses and countries, and extract postal codes.
2. **Candidate generation.** Use TF-IDF similarity on names and exact postal code matches to shortlist likely
   matches for each record, instead of comparing every pair.
3. **Feature engineering.** Compute name, address and location similarity features for each candidate pair.
4. **Modeling.** Train a LightGBM classifier on the pair features, using a validation split grouped by Source 1 entity.
5. **Threshold tuning.** Choose the decision threshold by maximizing the competition's macro F0.5 metric directly.
6. **Inference and validation.** Predict on the test set, write the submission files and run consistency checks.

Only the challenge data is used: no pretrained models, external datasets or lookup APIs.

> Update the paths in the configuration cell before running.

In [ ]:
# Install dependencies not preinstalled on Kaggle
!pip install -q sparse_dot_topn rapidfuzz lightgbm

In [ ]:
# Imports and configuration
import os, re, gc, unicodedata
import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sparse_dot_topn import awesome_cossim_topn
from rapidfuzz import fuzz
import lightgbm as lgb

pd.set_option('display.max_colwidth', 120)
np.random.seed(42)

# Data locations (update to match your environment)
BASE_TRAIN = "/kaggle/input/datasets/kmps07/amazonml2026"
BASE_TEST  = "/kaggle/input/datasets/kmps07/amazontest"

PATHS = {
    'train_s1': f"{BASE_TRAIN}/train_source1.tsv",
    'train_s2': f"{BASE_TRAIN}/train_source2.tsv",
    'train_s3': f"{BASE_TRAIN}/train_source3.tsv",
    'train_gt': f"{BASE_TRAIN}/train_ground_truth.tsv",
    'test_s1':  f"{BASE_TEST}/test_source1.tsv",
    'test_s2':  f"{BASE_TEST}/test_source2.tsv",
    'test_s3':  f"{BASE_TEST}/test_source3.tsv",
}

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Candidate generation parameters
TOPN = 15                  # candidates kept per Source 1 row
LOWER_BOUND = 0.15         # minimum cosine similarity to keep a candidate
ORPHAN_LOWER_BOUND = 0.07  # relaxed threshold for the orphan fallback pass
MAX_TFIDF_FEATURES = 200_000

## 1. Load data

The files are tab-separated. Everything is read as raw strings (`dtype=str`) and empty fields stay as empty
strings instead of `NaN`, which keeps loading fast and makes the text processing simpler.

In [ ]:
def load_tsv(path):
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False, na_filter=False)

train_s1 = load_tsv(PATHS['train_s1'])
train_s2 = load_tsv(PATHS['train_s2'])
train_s3 = load_tsv(PATHS['train_s3'])
train_gt = load_tsv(PATHS['train_gt'])

test_s1 = load_tsv(PATHS['test_s1'])
test_s2 = load_tsv(PATHS['test_s2'])
test_s3 = load_tsv(PATHS['test_s3'])

print("train:", train_s1.shape, train_s2.shape, train_s3.shape, train_gt.shape)
print("test :", test_s1.shape, test_s2.shape, test_s3.shape)

## 2. Normalization

- **Names:** remove accents, lowercase, replace `&` with `and`, remove punctuation, and drop common legal suffixes
  (Inc, Corp, Ltd, Pvt, LLC, ...) to get a "core" name. The full normalized name (suffixes kept) is also stored as a
  second similarity signal.
- **Addresses:** lowercase, remove accents, drop `null` / `n/a` tokens, expand common abbreviations (`rd` to `road`,
  `st` to `street`, ...), and extract a postal code with a regex. The postal code does not depend on the script,
  so it still works when the business name is transliterated.
- **Country:** treated as an open-set string (trimmed and lowercased) and never hard-coded to a fixed list, so
  countries that appear only in the test data, such as France, are handled the same way.

In [ ]:
LEGAL_SUFFIXES = set('''
inc incorporated corp corporation co company llc ltd limited pvt private plc llp
group holdings holding enterprises enterprise industries industry associates associate
partners partner pc pa lp trust foundation society services service solutions ventures
venture systems technologies technology international national global
'''.split())

ADDR_ABBR = {
    'rd':'road','st':'street','ave':'avenue','blvd':'boulevard','dr':'drive','ln':'lane',
    'ct':'court','cir':'circle','hwy':'highway','apt':'apartment','fl':'floor','no':'number',
    'ph':'phase','nr':'near','opp':'opposite','sec':'sector','twp':'township','ext':'extension',
}

def strip_accents(s):
    return ''.join(c for c in unicodedata.normalize('NFKD', s) if not unicodedata.combining(c))

def normalize_name(name):
    if not name:
        return "", ""
    s = strip_accents(str(name)).lower()
    s = s.replace('&', ' and ')
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    full = s
    tokens = [t for t in s.split() if t not in LEGAL_SUFFIXES]
    core = ' '.join(tokens) if tokens else s
    return core, full

def normalize_address(addr):
    if not addr:
        return "", ""
    raw = str(addr)
    s = strip_accents(raw).lower()
    s = re.sub(r'\bnull\b', ' ', s)
    s = re.sub(r'\bn a\b', ' ', s)
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    tokens = [ADDR_ABBR.get(t, t) for t in s.split()]
    norm = ' '.join(tokens)
    postal_matches = re.findall(r'\b\d{5,6}\b', raw)
    postal = postal_matches[-1] if postal_matches else ""
    return norm, postal

def normalize_country(c):
    if not c:
        return "unknown"
    return re.sub(r'\s+', ' ', str(c).strip().lower())

def preprocess(df):
    names = df['business_name'].map(normalize_name)
    addrs = df['business_address'].map(normalize_address)
    return pd.DataFrame({
        'entity_id':    df['entity_id'].values,
        'name_core':    [t[0] for t in names],
        'name_full':    [t[1] for t in names],
        'addr_norm':    [t[0] for t in addrs],
        'postal':       [t[1] for t in addrs],
        'country_norm': df['country'].map(normalize_country).values,
    })

train_s1p = preprocess(train_s1)
train_s2p = preprocess(train_s2)
train_s3p = preprocess(train_s3)
test_s1p  = preprocess(test_s1)
test_s2p  = preprocess(test_s2)
test_s3p  = preprocess(test_s3)

# Free the raw dataframes; only the normalized columns are needed from here on
del train_s1, train_s2, train_s3, test_s1, test_s2, test_s3
gc.collect()

print(train_s1p.head(3))

## 3. Candidate generation

Comparing every Source 1 record with every Source 2/3 record would be too slow, so a shortlist of likely matches
is generated for each record:

- **Name similarity.** Within each country, names are vectorized with 2-4 character n-gram TF-IDF, and the top
  `TOPN` most similar records are kept per row using sparse cosine similarity.
- **Postal code match.** Records with the same country and postal code are added as candidates, which helps when
  names are written in different scripts.
- **Fallback.** Source 1 rows with no candidates get a second, relaxed search across countries.

The share of true matches that reach this shortlist is checked below, since the model can only choose among the
candidates it is given.

In [ ]:
def fit_vectorizer(*dfs, max_features=MAX_TFIDF_FEATURES):
    corpus = pd.concat([d['name_core'] for d in dfs], ignore_index=True)
    corpus = corpus[corpus.str.len() > 0]
    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                          max_features=max_features, dtype=np.float32)
    vec.fit(corpus.values)
    return vec

def tfidf_topn_candidates(left_df, right_df, vectorizer, topn, lower_bound):
    if len(left_df) == 0 or len(right_df) == 0:
        return pd.DataFrame(columns=['source1_entity_id', 'candidate_entity_id', 'sim_score'])
    A = vectorizer.transform(left_df['name_core'].values).astype(np.float32)
    B = vectorizer.transform(right_df['name_core'].values).astype(np.float32)
    C = awesome_cossim_topn(A, B.transpose(), topn, lower_bound)
    C = C.tocoo()
    left_ids = left_df['entity_id'].values
    right_ids = right_df['entity_id'].values
    out = pd.DataFrame({
        'source1_entity_id':   left_ids[C.row],
        'candidate_entity_id': right_ids[C.col],
        'sim_score':           C.data.astype(np.float32),
    })
    del A, B, C
    gc.collect()
    return out

def pincode_candidates(left_df, right_df):
    left = left_df[left_df['postal'] != ''][['entity_id', 'country_norm', 'postal']] \
        .rename(columns={'entity_id': 'source1_entity_id'})
    right = right_df[right_df['postal'] != ''][['entity_id', 'country_norm', 'postal']] \
        .rename(columns={'entity_id': 'candidate_entity_id'})
    if len(left) == 0 or len(right) == 0:
        return pd.DataFrame(columns=['source1_entity_id', 'candidate_entity_id', 'sim_score'])
    merged = left.merge(right, on=['country_norm', 'postal'], how='inner')
    merged['sim_score'] = np.float32(0.5)  # placeholder score; the postal signal is captured by f_postal_match
    return merged[['source1_entity_id', 'candidate_entity_id', 'sim_score']]

def generate_candidates(s1p, s2p, s3p, vectorizer, topn=TOPN, lower_bound=LOWER_BOUND):
    other = pd.concat([s2p, s3p], ignore_index=True)

    # Step 1: TF-IDF candidates within each country
    parts = []
    countries = sorted(set(s1p['country_norm']) | set(other['country_norm']))
    for country in countries:
        left = s1p[s1p['country_norm'] == country]
        right = other[other['country_norm'] == country]
        if len(left) == 0 or len(right) == 0:
            continue
        parts.append(tfidf_topn_candidates(left, right, vectorizer, topn, lower_bound))

    cand_df = pd.concat(parts, ignore_index=True) if parts else \
        pd.DataFrame(columns=['source1_entity_id', 'candidate_entity_id', 'sim_score'])
    del parts
    gc.collect()

    # Step 2: relaxed cross-country search for Source 1 rows with no candidates
    matched_s1 = set(cand_df['source1_entity_id'].unique())
    orphans = s1p[~s1p['entity_id'].isin(matched_s1)]
    if len(orphans) > 0:
        extra = tfidf_topn_candidates(orphans, other, vectorizer, topn, ORPHAN_LOWER_BOUND)
        cand_df = pd.concat([cand_df, extra], ignore_index=True)
        del extra
        gc.collect()

    # Step 3: exact postal code candidates
    pin_cand = pincode_candidates(s1p, other)
    cand_df = pd.concat([cand_df, pin_cand], ignore_index=True)
    del pin_cand, other
    gc.collect()

    # Remove duplicate pairs, keeping the highest similarity score
    cand_df = cand_df.groupby(['source1_entity_id', 'candidate_entity_id'], as_index=False)['sim_score'].max()
    return cand_df

In [ ]:
# Generate training candidates and check how many true matches they cover
train_vectorizer = fit_vectorizer(train_s1p, train_s2p, train_s3p)
train_candidates = generate_candidates(train_s1p, train_s2p, train_s3p, train_vectorizer)
print("Train candidate pairs:", len(train_candidates))

gt_pairs = train_gt[train_gt['matched_entity_ids'] != ''].copy()
gt_pairs['matched_list'] = gt_pairs['matched_entity_ids'].str.split(',')
gt_exploded = gt_pairs.explode('matched_list').rename(columns={'matched_list': 'candidate_entity_id'})
gt_pair_set = set(zip(gt_exploded['source1_entity_id'], gt_exploded['candidate_entity_id']))

cand_pair_set = set(zip(train_candidates['source1_entity_id'], train_candidates['candidate_entity_id']))
hit = len(gt_pair_set & cand_pair_set)
recall_ceiling = hit / max(1, len(gt_pair_set))
n_other = len(train_s2p) + len(train_s3p)
reduction_ratio = 1 - len(train_candidates) / max(1, len(train_s1p) * n_other)

print(f"Candidate recall           : {recall_ceiling:.4f}  ({hit}/{len(gt_pair_set)})")
print(f"Reduction vs. all pairs      : {reduction_ratio:.6f}")

## 4. Feature engineering

For every candidate pair, the following features are computed:

- RapidFuzz similarity scores and token/character Jaccard similarity on the core and full normalized names
- Similarity scores on the normalized address
- Exact postal code match and country match flags
- The cosine similarity score from candidate generation

In [ ]:
def jaccard_tokens(a, b):
    sa, sb = set(a.split()), set(b.split())
    if not sa and not sb: return 1.0
    if not sa or not sb: return 0.0
    return len(sa & sb) / len(sa | sb)

def char_bigram_jaccard(a, b):
    def bigr(s):
        return set(s[i:i+2] for i in range(len(s) - 1)) if len(s) > 1 else ({s} if s else set())
    ba, bb = bigr(a), bigr(b)
    if not ba and not bb: return 1.0
    if not ba or not bb: return 0.0
    return len(ba & bb) / len(ba | bb)

def common_prefix_len(a, b):
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i

def build_pair_features(cand_df, s1p, otherp):
    left = s1p.set_index('entity_id')[['name_core', 'name_full', 'addr_norm', 'postal', 'country_norm']]
    right = otherp.set_index('entity_id')[['name_core', 'name_full', 'addr_norm', 'postal', 'country_norm']]

    df = cand_df.merge(left, left_on='source1_entity_id', right_index=True, how='left')
    df = df.rename(columns={c: f's1_{c}' for c in left.columns})
    df = df.merge(right, left_on='candidate_entity_id', right_index=True, how='left')
    df = df.rename(columns={c: f's2_{c}' for c in right.columns})

    for col in ['s1_name_core','s2_name_core','s1_name_full','s2_name_full',
                's1_addr_norm','s2_addr_norm','s1_postal','s2_postal',
                's1_country_norm','s2_country_norm']:
        df[col] = df[col].fillna('')

    n1, n2   = df['s1_name_core'].values, df['s2_name_core'].values
    fn1, fn2 = df['s1_name_full'].values, df['s2_name_full'].values
    a1, a2   = df['s1_addr_norm'].values, df['s2_addr_norm'].values

    # Name features
    df['f_name_ratio']        = [fuzz.ratio(x, y) / 100.0 for x, y in zip(n1, n2)]
    df['f_name_tsort']        = [fuzz.token_sort_ratio(x, y) / 100.0 for x, y in zip(n1, n2)]
    df['f_name_tset']         = [fuzz.token_set_ratio(x, y) / 100.0 for x, y in zip(n1, n2)]
    df['f_name_partial']      = [fuzz.partial_ratio(x, y) / 100.0 for x, y in zip(n1, n2)]
    df['f_name_jaccard']      = [jaccard_tokens(x, y) for x, y in zip(n1, n2)]
    df['f_name_bigram_jac']   = [char_bigram_jaccard(x, y) for x, y in zip(n1, n2)]
    df['f_name_full_ratio']   = [fuzz.ratio(x, y) / 100.0 for x, y in zip(fn1, fn2)]
    df['f_name_len_diff']     = [abs(len(x) - len(y)) / max(len(x), len(y), 1) for x, y in zip(n1, n2)]
    df['f_name_prefix']       = [common_prefix_len(x, y) / max(len(x), len(y), 1) for x, y in zip(n1, n2)]
    df['f_first_token_match'] = [
        (x.split()[0] if x.split() else '') == (y.split()[0] if y.split() else '')
        for x, y in zip(n1, n2)
    ]
    df['f_first_token_match'] = df['f_first_token_match'].astype(np.int8)

    # Address features
    df['f_addr_ratio']      = [fuzz.token_sort_ratio(x, y) / 100.0 for x, y in zip(a1, a2)]
    df['f_addr_jaccard']    = [jaccard_tokens(x, y) for x, y in zip(a1, a2)]
    df['f_addr_bigram_jac'] = [char_bigram_jaccard(x, y) for x, y in zip(a1, a2)]

    # Structured features
    df['f_postal_match']  = ((df['s1_postal'] != '') & (df['s1_postal'] == df['s2_postal'])).astype(np.int8)
    df['f_country_match'] = (df['s1_country_norm'] == df['s2_country_norm']).astype(np.int8)
    df['f_sim_score']     = df['sim_score'].astype(np.float32)

    feature_cols = [c for c in df.columns if c.startswith('f_')]
    keep_cols = ['source1_entity_id', 'candidate_entity_id'] + feature_cols
    return df[keep_cols], feature_cols

In [ ]:
# Build features and labels for the training candidates
train_other_p = pd.concat([train_s2p, train_s3p], ignore_index=True)
train_feat_df, FEATURE_COLS = build_pair_features(train_candidates, train_s1p, train_other_p)

pair_key = list(zip(train_feat_df['source1_entity_id'], train_feat_df['candidate_entity_id']))
train_feat_df['label'] = np.fromiter((p in gt_pair_set for p in pair_key), dtype=np.int8, count=len(pair_key))
del pair_key, train_other_p
gc.collect()

print(FEATURE_COLS)
print(train_feat_df['label'].value_counts())

## 5. Train/validation split and LightGBM model

The data is split by Source 1 entity ID, so all candidate pairs of an entity stay in the same split. This avoids
leakage and matches how the metric is computed (averaged per Source 1 entity).

In [ ]:
all_s1_ids = train_s1p['entity_id'].unique()
train_ids, val_ids = train_test_split(all_s1_ids, test_size=0.15, random_state=42)
train_ids, val_ids = set(train_ids), set(val_ids)

tr_mask  = train_feat_df['source1_entity_id'].isin(train_ids)
val_mask = train_feat_df['source1_entity_id'].isin(val_ids)

X_train, y_train = train_feat_df.loc[tr_mask, FEATURE_COLS],  train_feat_df.loc[tr_mask, 'label']
X_val,   y_val   = train_feat_df.loc[val_mask, FEATURE_COLS], train_feat_df.loc[val_mask, 'label']

print(X_train.shape, X_val.shape, "positive rate (train/val):", y_train.mean(), y_val.mean())

lgb_train = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
lgb_val   = lgb.Dataset(X_val,   label=y_val,   reference=lgb_train, free_raw_data=False)

params = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'max_depth': -1,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 3,
    'min_data_in_leaf': 20,
    'is_unbalance': True,
    'verbosity': -1,
    'seed': 42,
}

model = lgb.train(
    params, lgb_train,
    num_boost_round=3000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(80), lgb.log_evaluation(100)],
)

lgb.plot_importance(model, max_num_features=20, figsize=(7, 6), importance_type='gain')

## 6. Threshold search on the competition metric

The decision threshold is not chosen from AUC or log loss. Instead, the competition metric is implemented directly
(macro-averaged F0.5 per Source 1 entity, where a correctly predicted empty match list scores 1.0) and the
threshold is swept against it, since that is what the final score is based on.

In [ ]:
def macro_f0_5(pred_dict, truth_dict, s1_ids):
    scores = []
    for sid in s1_ids:
        pred_set = pred_dict.get(sid, set())
        true_set = truth_dict.get(sid, set())
        if len(true_set) == 0 and len(pred_set) == 0:
            scores.append(1.0)
        elif len(true_set) == 0 or len(pred_set) == 0:
            scores.append(0.0)
        else:
            tp = len(pred_set & true_set)
            precision = tp / len(pred_set)
            recall = tp / len(true_set)
            if precision == 0 and recall == 0:
                scores.append(0.0)
            else:
                scores.append((1.25 * precision * recall) / (0.25 * precision + recall))
    return float(np.mean(scores))

val_pred_df = train_feat_df.loc[val_mask, ['source1_entity_id', 'candidate_entity_id']].copy()
val_pred_df['prob'] = model.predict(X_val, num_iteration=model.best_iteration)

val_gt_sub = train_gt[train_gt['source1_entity_id'].isin(val_ids)]
val_truth = {}
for sid, matched in zip(val_gt_sub['source1_entity_id'], val_gt_sub['matched_entity_ids']):
    val_truth[sid] = set(matched.split(',')) if matched else set()

best_th, best_f = 0.5, -1.0
curve = []
for th in np.arange(0.20, 0.97, 0.01):
    sub = val_pred_df[val_pred_df['prob'] >= th]
    pred_dict = sub.groupby('source1_entity_id')['candidate_entity_id'].apply(set).to_dict()
    f = macro_f0_5(pred_dict, val_truth, val_ids)
    curve.append((th, f))
    if f > best_f:
        best_f, best_th = f, th

print(f"Best threshold = {best_th:.2f}   validation F0.5 = {best_f:.4f}")
for th, f in curve[::5]:
    print(f"  threshold={th:.2f}  F0.5={f:.4f}")

## 7. Inference on the test set and submission files

A new TF-IDF vectorizer is fitted on the test data, since the test set includes France, a country not present in
training. The classifier only uses generic similarity features scaled between 0 and 1, so it transfers to the new
data. Candidate generation and feature engineering use the same functions as in training.

In [ ]:
test_vectorizer = fit_vectorizer(test_s1p, test_s2p, test_s3p)
test_candidates = generate_candidates(test_s1p, test_s2p, test_s3p, test_vectorizer)
print("Test candidate pairs:", len(test_candidates))

test_other_p = pd.concat([test_s2p, test_s3p], ignore_index=True)
test_feat_df, _ = build_pair_features(test_candidates, test_s1p, test_other_p)
del test_other_p
gc.collect()

test_feat_df['prob'] = model.predict(test_feat_df[FEATURE_COLS], num_iteration=model.best_iteration)
gc.collect()

In [ ]:
FINAL_THRESHOLD = best_th  # can be overridden manually to trade off precision and recall

def build_output(cand_feat_df, s1_ids, threshold, id_col, score_col='prob'):
    matched = cand_feat_df[cand_feat_df[score_col] >= threshold]
    grouped = matched.groupby('source1_entity_id')[id_col].apply(lambda s: ','.join(sorted(set(s))))
    out = pd.DataFrame({'source1_entity_id': s1_ids})
    out = out.merge(grouped.rename('matched_entity_ids'), left_on='source1_entity_id', right_index=True, how='left')
    out['matched_entity_ids'] = out['matched_entity_ids'].fillna('')
    return out

test_s1_ids_all = test_s1p['entity_id'].tolist()

matching_results = build_output(test_feat_df, test_s1_ids_all, FINAL_THRESHOLD, id_col='candidate_entity_id')
matching_results.to_csv(f"{OUTPUT_DIR}/matching_results.tsv", sep='\t', index=False)

cand_out = test_candidates.groupby('source1_entity_id')['candidate_entity_id'] \
    .apply(lambda s: ','.join(sorted(set(s))))
candidate_pairs_out = pd.DataFrame({'source1_entity_id': test_s1_ids_all})
candidate_pairs_out = candidate_pairs_out.merge(
    cand_out.rename('candidate_entity_ids'), left_on='source1_entity_id', right_index=True, how='left')
candidate_pairs_out['candidate_entity_ids'] = candidate_pairs_out['candidate_entity_ids'].fillna('')
candidate_pairs_out.to_csv(f"{OUTPUT_DIR}/candidate_pairs.tsv", sep='\t', index=False)

print(matching_results.shape, candidate_pairs_out.shape)
matching_results.head()

## 8. Submission checks

These checks mirror the official validation script: row counts, duplicate rows, valid entity IDs, and that every
predicted match is also in the candidate set.

In [ ]:
assert len(matching_results) == len(test_s1_ids_all), "row count mismatch"
assert matching_results['source1_entity_id'].nunique() == len(matching_results), "duplicate source1 rows"
assert set(matching_results['source1_entity_id']) == set(test_s1_ids_all), "missing or extra source1 ids"

valid_ids = set(test_s2p['entity_id']) | set(test_s3p['entity_id'])
issues = 0
for lst in matching_results['matched_entity_ids']:
    if not lst:
        continue
    ids = lst.split(',')
    if len(ids) != len(set(ids)):
        issues += 1
    if not all(i in valid_ids for i in ids):
        issues += 1

print("Rows with issues:", issues)
print("Non-empty match rows:", (matching_results['matched_entity_ids'] != '').sum(), "/", len(matching_results))

# Every predicted match must also appear in the candidate set
match_pairs = set()
for sid, lst in zip(matching_results['source1_entity_id'], matching_results['matched_entity_ids']):
    for cid in (lst.split(',') if lst else []):
        match_pairs.add((sid, cid))
cand_pairs_test = set(zip(test_candidates['source1_entity_id'], test_candidates['candidate_entity_id']))
orphan_matches = match_pairs - cand_pairs_test
print("Predicted matches missing from the candidate set (expected 0):", len(orphan_matches))

In [ ]:
# Optional: run the official validator if it is available in this environment
import subprocess

VALIDATOR = "utils/validate_submission.py"  # path to the validator provided with the challenge files
if os.path.exists(VALIDATOR):
    result = subprocess.run(
        ["python3", VALIDATOR,
         "--matching", f"{OUTPUT_DIR}/matching_results.tsv",
         "--candidate", f"{OUTPUT_DIR}/candidate_pairs.tsv",
         "--test-dir", os.path.dirname(PATHS['test_s1'])],
        capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
else:
    print(f"Validator not found at '{VALIDATOR}'. Copy the challenge's validate_submission.py "
          "into the working directory to run it.")

## 9. Possible improvements

1. **Error analysis.** Review false positives at the chosen threshold. They often follow a pattern, such as
   different branches of the same chain or very short generic names with little address overlap. A targeted
   feature for that pattern could reduce them.
2. **Class weighting.** Replacing `is_unbalance=True` with a lower, tuned `scale_pos_weight` can push the model
   toward higher precision, which suits the F0.5 metric.
3. **Ensembling.** Averaging LightGBM with another gradient-boosted model (CatBoost or XGBoost) trained on the
   same features may improve results.
4. **Candidate coverage.** If many true matches are missing from the shortlist (section 3), increasing `TOPN` or
   lowering `LOWER_BOUND` can help, at the cost of more memory and time.
5. **Memory use.** Features use `float32` and `int8` types, and intermediate dataframes are deleted after use, so
   the pipeline can run on a standard Kaggle notebook.